# JuaKazi `zu-bias-classifier-v1` — Training Notebook

**Model:** `juakazike/zu-bias-classifier-v1`  
**Base:** `Davlan/afro-xlmr-base` · **Task:** Zulu gender bias (binary)  
**Runtime:** GPU T4 · ~1 hour  
**Target:** BIAS Precision ≥ 0.60, Recall ≥ 0.65, F1 ≥ 0.68

### Data sources (~14,570 rows after merge)

| File | Rows | Labels |
|---|---|---|
| `IsiZulu_Ithute_Dataset_Final.csv` | 9,570 | stereotype + derogation (all passed) |
| `eval/ground_truth_zu_v1.csv` | 2,000 | biased (1,978) + neutral (22) |
| `data/neutral_zu_v1.csv` | 3,000 | neutral (generated from CC-100) |

**Label mapping:**
- `stereotype` + `derogation` → **BIASED (1)**
- `neutral` → **NEUTRAL (0)**

### Key challenge
Original ZU data is 99% biased. The `neutral_zu_v1.csv` file provides balance.  
Train/val/test split is 70/15/15 (larger test set for small dataset).

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload all 3 CSV files to Drive at `MyDrive/juakazi/`
3. Add `HF_TOKEN` to Colab Secrets


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'transformers>=4.38.0', 'tokenizers>=0.15.0', 'accelerate>=0.27.0',
    'scikit-learn>=1.4.0', 'huggingface_hub>=0.20.0', 'numpy<2.0.0',
], capture_output=True, text=True)
print('Install OK.' if result.returncode == 0 else f'FAILED:\n{result.stderr}')
print('Restart runtime, then run from Cell 2.')

In [ ]:
# ── Cell 2: Verify environment ───────────────────────────────────────────────
import torch, transformers, sklearn, numpy as np
print(f'torch: {torch.__version__}  |  transformers: {transformers.__version__}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No GPU — go to Runtime → Change runtime type → T4 GPU'

In [ ]:
# ── Cell 3: Mount Drive + load data ─────────────────────────────────────────
# Kaggle: files at /kaggle/input/{dataset-slug}/

import os, csv
BASE_DIR = '/kaggle/input/juakazi-zu-training'

ITHUTE_CSV  = f'{BASE_DIR}/IsiZulu_Ithute_Dataset_Final.csv'
GT_CSV      = f'{BASE_DIR}/ground_truth_zu_v1.csv'
NEUTRAL_CSV = f'{BASE_DIR}/neutral_zu_v1.csv'

for p in [ITHUTE_CSV, NEUTRAL_CSV]:
    assert os.path.exists(p), f'Not found: {p}'

def read_csv(path):
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))

ithute_rows  = read_csv(ITHUTE_CSV)
gt_rows      = read_csv(GT_CSV) if os.path.exists(GT_CSV) else []
neutral_rows = read_csv(NEUTRAL_CSV)

print(f'Ithute:  {len(ithute_rows):,} rows')
print(f'GT:      {len(gt_rows):,} rows')
print(f'Neutral: {len(neutral_rows):,} rows')

In [ ]:
# ── Cell 4: Label mapping + merge + dedup ────────────────────────────────────
BIASED_LABELS  = {'stereotype', 'derogation'}

all_data   = []
seen_texts = set()

# IsiZulu Ithute — text column is 'Zulu text'
for r in ithute_rows:
    text = r.get('Zulu text', r.get('text', '')).strip()
    bl   = r.get('bias_label', '').lower().strip()
    if not text or text in seen_texts: continue
    if bl in BIASED_LABELS:
        seen_texts.add(text)
        all_data.append((text, 1))

# GT — has has_bias boolean
for r in gt_rows:
    text = r.get('text', '').strip()
    hb   = str(r.get('has_bias', '')).strip().lower()
    if not text or text in seen_texts: continue
    seen_texts.add(text)
    all_data.append((text, 1 if hb in ('true', '1', 'yes') else 0))

# Neutral CC-100 rows
for r in neutral_rows:
    text = r.get('text', '').strip()
    if not text or text in seen_texts: continue
    seen_texts.add(text)
    all_data.append((text, 0))

bias_data    = [(t, l) for t, l in all_data if l == 1]
neutral_data = [(t, l) for t, l in all_data if l == 0]
n_bias, n_neutral = len(bias_data), len(neutral_data)

print(f'Total unique: {len(all_data):,}')
print(f'Biased:  {n_bias:,}  |  Neutral: {n_neutral:,}')
print(f'Ratio: {n_bias/max(n_neutral,1):.1f}:1 biased')

In [ ]:
# ── Cell 5: Config ───────────────────────────────────────────────────────────
import random, numpy as np
from pathlib import Path

SEED          = 42
BASE_MODEL    = 'Davlan/afro-xlmr-base'
OUTPUT_DIR    = '/kaggle/working/output'
MAX_LEN       = 128
# ZU is majority biased — inverse ratio is < 1, so POS_WEIGHT = 1.0 means equal weight
# But we may have more biased than neutral, so weight neutral higher
raw_ratio     = n_neutral / max(n_bias, 1)
POS_WEIGHT    = max(1.0 / max(raw_ratio, 0.01), 1.0)  # weight bias class if underrepresented
POS_WEIGHT    = min(POS_WEIGHT, 15.0)                  # hard cap
TRAIN_SPLIT   = 0.70
VAL_SPLIT     = 0.15
EPOCHS        = 10
BATCH         = 16
LR            = 2e-5
WARMUP_RATIO  = 0.10
WEIGHT_DECAY  = 0.01
FREEZE_LAYERS = 6
REPO_ID       = 'juakazike/zu-bias-classifier-v1'

random.seed(SEED); np.random.seed(SEED)
import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'n_bias={n_bias:,}  n_neutral={n_neutral:,}  raw_ratio={raw_ratio:.2f}  POS_WEIGHT={POS_WEIGHT:.2f}')
print(f'REPO={REPO_ID}')

In [ ]:
# ── Cell 6: Build train/val/test splits ──────────────────────────────────────
import random

combined = all_data[:]
random.shuffle(combined)

n       = len(combined)
n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)

train_data = combined[:n_train]
val_data   = combined[n_train:n_train + n_val]
test_data  = combined[n_train + n_val:]

print(f'Train: {len(train_data):,}  ({sum(1 for _,l in train_data if l==1):,} bias)')
print(f'Val:   {len(val_data):,}  ({sum(1 for _,l in val_data   if l==1):,} bias)')
print(f'Test:  {len(test_data):,}  ({sum(1 for _,l in test_data  if l==1):,} bias)  ← held out')

In [ ]:
# ── Cell 7: Tokenizer + Dataset ──────────────────────────────────────────────
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class BiasDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data; self.tokenizer = tokenizer; self.max_len = max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer(text, truncation=True, max_length=self.max_len,
                             padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}

train_ds = BiasDataset(train_data, tokenizer, MAX_LEN)
val_ds   = BiasDataset(val_data,   tokenizer, MAX_LEN)
test_ds  = BiasDataset(test_data,  tokenizer, MAX_LEN)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
# ── Cell 8: Model + WeightedTrainer ─────────────────────────────────────────
import torch, numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0: 'NEUTRAL', 1: 'BIAS'}, label2id={'NEUTRAL': 0, 'BIAS': 1},
    ignore_mismatched_sizes=True,
)
for i, layer in enumerate(model.roberta.encoder.layer):
    if i < FREEZE_LAYERS:
        for p in layer.parameters(): p.requires_grad = False

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weight = torch.tensor([1.0, POS_WEIGHT], dtype=torch.float, device=outputs.logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weight)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = int(((preds==1)&(labels==1)).sum()); fp = int(((preds==1)&(labels==0)).sum())
    fn = int(((preds==0)&(labels==1)).sum())
    p = tp/(tp+fp) if (tp+fp)>0 else 0; r = tp/(tp+fn) if (tp+fn)>0 else 0
    f = 2*p*r/(p+r) if (p+r)>0 else 0
    print(f'  TP={tp} FP={fp} FN={fn} | P={p:.3f} R={r:.3f} F1={f:.3f}')
    return {'f1': f, 'precision': p, 'recall': r}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    learning_rate=LR, warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1', greater_is_better=True, logging_steps=50,
    fp16=torch.cuda.is_available(), seed=SEED, report_to='none',
)
trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print('Trainer ready. POS_WEIGHT=', POS_WEIGHT)

In [ ]:
# ── Cell 9: TRAIN ───────────────────────────────────────────────────────────
result = trainer.train()
print(f'Done. Steps={result.global_step}  Loss={result.training_loss:.4f}')

In [ ]:
# ── Cell 10: Evaluate on TEST SET ────────────────────────────────────────────
import numpy as np, torch, torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score

pred_out = trainer.predict(test_ds)
preds    = np.argmax(pred_out.predictions, axis=-1)
labels   = pred_out.label_ids

print('=== TEST SET RESULTS ===')
print(classification_report(labels, preds, target_names=['NEUTRAL', 'BIAS']))

# Optimal threshold
probs   = F.softmax(torch.tensor(pred_out.predictions), dim=-1)[:, 1].numpy()
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.20, 0.90, 0.01):
    f = f1_score(labels, (probs >= t).astype(int), average='binary')
    if f > best_f1: best_f1, best_t = f, float(t)

final_f1   = best_f1
target_met = final_f1 >= 0.68
print(f'\nBest threshold: {best_t:.2f}  Test F1: {final_f1:.4f}  (target ≥0.68)  {"✓ TARGET MET" if target_met else "✗ BELOW TARGET"}')
print(f'>>> Set: JUAKAZI_ZU_THRESHOLD={best_t:.2f}')

if not target_met:
    print('\nIf below target: check neutral data quality, adjust POS_WEIGHT, or add more neutral rows.')

In [ ]:
# ── Cell 11: Save + upload to HuggingFace ────────────────────────────────────
import json
from sklearn.metrics import precision_score, recall_score
from huggingface_hub import HfApi
import os

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

opt_preds = (probs >= best_t).astype(int)
meta = {
    'model_id': REPO_ID, 'base_model': BASE_MODEL, 'language': 'zu',
    'test_f1': round(best_f1, 4),
    'test_precision': round(float(precision_score(labels, opt_preds, zero_division=0)), 4),
    'test_recall': round(float(recall_score(labels, opt_preds, zero_division=0)), 4),
    'threshold': best_t, 'target_met': target_met,
    'train_size': len(train_data), 'test_size': len(test_data),
    'n_bias': n_bias, 'n_neutral': n_neutral, 'pos_weight': POS_WEIGHT,
}
with open(f'{OUTPUT_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

try:
    HF_TOKEN = os.environ.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN and target_met:
    api = HfApi()
    api.create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True, private=False)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, token=HF_TOKEN)
    print(f'Uploaded → https://huggingface.co/{REPO_ID}')
    print(f'Set: JUAKAZI_ZU_MODEL={REPO_ID}  JUAKAZI_ZU_THRESHOLD={best_t:.2f}')
elif not target_met:
    print('⚠️  Not uploading — target not met.')
else:
    print('Set HF_TOKEN in Colab Secrets to upload.')